# TowerIQ Quarantine Layer Inspection

Use this notebook to inspect invalid records written to Quarantine.

For the current tiny clean dataset, quarantine counts are expected to be zero. This notebook will become more useful after bad-record injection is added.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))

from src.ingestion.bronze_ingestion import build_storage_path
from src.ingestion.schemas import RAW_SCHEMAS
from src.utils.config import load_config
from src.utils.spark import create_spark_session

config = load_config("configs/local.yaml")
spark_config = config["spark"]
paths = config["paths"]

spark = create_spark_session(
    app_name="TowerIQ-QuarantineInspection",
    master=spark_config["master"],
    aqe_enabled=bool(spark_config["adaptive_query_execution"]),
    use_pyspark_package=bool(spark_config.get("use_pyspark_package", True)),
)
spark

## Load Quarantine Tables

In [ ]:
quarantine_tables = {
    table_name: spark.read.parquet(build_storage_path(paths["quarantine"], "tiny", table_name))
    for table_name in RAW_SCHEMAS
}

for table_name, df in quarantine_tables.items():
    df.createOrReplaceTempView(f"quarantine_{table_name}")
    print(table_name, df.count())

## Q1. How many quarantined records exist per table?

In [ ]:
for table_name, df in quarantine_tables.items():
    print(f"{table_name}: {df.count():,}")

## Q2. Do quarantine tables contain rejection metadata?

In [ ]:
metadata_columns = ["_rejection_reasons", "_quarantined_at", "_source_table"]
for table_name, df in quarantine_tables.items():
    missing = [column for column in metadata_columns if column not in df.columns]
    print(table_name, "missing_metadata=", missing)

## Q3. Rejection reason summary

This will show records after bad-data injection is added.

In [ ]:
from pyspark.sql import functions as F

reason_summaries = []
for table_name, df in quarantine_tables.items():
    if df.count() > 0:
        summary = (
            df.select(F.explode("_rejection_reasons").alias("rejection_reason"))
            .groupBy("rejection_reason")
            .count()
            .withColumn("table_name", F.lit(table_name))
            .select("table_name", "rejection_reason", "count")
        )
        reason_summaries.append(summary)

if reason_summaries:
    combined = reason_summaries[0]
    for summary in reason_summaries[1:]:
        combined = combined.unionByName(summary)
    combined.orderBy("table_name", F.desc("count")).show(100, truncate=False)
else:
    print("No quarantined records found in the current tiny clean dataset.")

## Q4. Inspect quarantined network events

Expected to be empty until bad-record injection is added.

In [ ]:
quarantine_tables["network_events"].select(
    "event_id", "subscriber_id", "device_id", "tower_id", "network_type",
    "status", "_rejection_reasons", "_quarantined_at", "_source_table"
).show(20, truncate=False)

## Q5. Inspect any non-empty quarantine tables

In [ ]:
for table_name, df in quarantine_tables.items():
    count = df.count()
    if count > 0:
        print(f"Showing {table_name}: {count:,} quarantined records")
        df.show(20, truncate=False)

## Stop Spark

In [ ]:
spark.stop()